In [ ]:
%load_ext autoreload
%autoreload 2

# Linkcomm

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, r"c:\users\arthu\documents\swittz-epfl\phd\code\lib\site-packages")
sys.path.append('../')
from pygraphon.graphons import StepGraphon
from pygraphon.estimators import SmoothNetHist

from experiments.utils_generators import AssociativeFullyRandom
from pygraphon.norm import MseProbaEdge, MaeProbaEdge, AUCEdge, AUPRCEdge
from pygraphon.plots.plot_matrices import spy

from linkcomm import HLC


In [ ]:
import numpy as np
def adjacency_matrix_to_dict(A):
    """
    Converts an adjacency matrix (numpy array) to a dictionary representation.

    Parameters:
    A (numpy array): An n x n adjacency matrix where A[i, j] is 1 if there is an edge from node i to node j, else 0.

    Returns:
    dict: A dictionary where keys are node indices and values are sets of neighbor indices.
    """
    A_dict = {}
    for i in range(A.shape[0]):
        A_dict[i] = set(np.where(A[i] == 1)[0])
    return A_dict

In [ ]:
def adjacency_matrix_to_edge_list(A):
    """
    Converts an adjacency matrix to an edge list.

    Parameters:
    A (numpy array): An n x n adjacency matrix where A[i, j] is 1 if there is an edge from node i to node j, else 0.

    Returns:
    set: A set of tuples representing edges. Each tuple is (i, j) where there is an edge from node i to node j.
    """
    edges = set()
    for i in range(A.shape[0]):
        for j in range(A.shape[1]):
            if A[i, j] == 1:
                edges.add((i, j))
    return edges

In [ ]:
from experiments.utils_generators import generate_hierarchical_theta

# Example usage:
k = 9
m = 3
theta_min_intra = 0.8
theta_max_intra = 0.8
theta_min_meta_intra = 0.9
theta_max_meta_intra = 0.9
theta_min_inter = 0.1
theta_max_inter = 0.1

theta = generate_hierarchical_theta(k, m, theta_min_intra, theta_max_intra, theta_min_meta_intra, theta_max_meta_intra, theta_min_inter, theta_max_inter)
seed_value = 42
np.random.seed(seed_value)
n = 200
graphon = StepGraphon(theta, bandwidthHist=1 /k)
adj_mat = graphon.draw(n=n, exchangeable=False)
adj_mat_dict = adjacency_matrix_to_dict(adj_mat)
edge_list = adjacency_matrix_to_edge_list(adj_mat)

In [ ]:
spy(adj_mat)

In [ ]:
edge2cid,S_max,D_max,list_D = HLC( adj_mat_dict,edge_list ).single_linkage()

In [ ]:
from linkcomm import similarities_unweighted
similarities_unweighted(adj_mat_dict)

In [ ]:
np.unique([x for x in edge2cid.values()])

In [ ]:
def convert_cid_to_adj(edge2cid, num_nodes):
    n =num_nodes
    adj = np.zeros((n,n))
    for e,cid in edge2cid.items():
        i,j = e
        adj[i,j] = cid
        adj[j,i] = cid
    return adj

In [ ]:
adj_viz = convert_cid_to_adj(edge2cid,n)

In [ ]:
len(np.unique(adj_viz))/len(edge2cid)

In [ ]:
plt.imshow(adj_viz)
plt.colorbar()
plt.show()

# Extract data from Marzia

In [ ]:
base_path_surv = "C:\\Users\\verdeyme\\Desktop\\Marzia\\marzia_data\\data_anonym\\surv"
# list of file name from p1 to p52 using for loop
folders_surv = ["p1", "p2", "p3", "p4", "p5", "p6", "p7", "p8", "p9", "p10", "p11", "p12", "p13", "p14", "p15", "p16", "p17", "p18", "p19", "p20", "p21", "p22", "p23", "p24", "p25", "p26", "p27", "p28", "p29", "p30", "p31", "p32", "p33", "p34", "p35", "p36", "p37", "p38", "p39", "p40", "p41", "p42", "p43", "p44", "p45", "p46", "p47", "p48", "p49", "p50", "p51", "p52"]
#copilot go brrrrrrr
base_path_nsruv = "C:\\Users\\verdeyme\\Desktop\\Marzia\\marzia_data\\data_anonym\\nonsurv"
folders_nsurv = folders_surv

mat_files = ["dwPLI_1_40Hz"]#, "dwPLI_10Hz_5trl_blocks_interp1st"]

In [ ]:
import scipy.io
import pandas as pd
import os

dataframes_wpli = {}

for folder in folders_surv:
    folder_path = os.path.join(base_path_surv, folder)
    for mat_file in mat_files:
        file_path = os.path.join(folder_path, mat_file + ".mat")
        if os.path.exists(file_path):
            mat_data = scipy.io.loadmat(file_path)
            data_key = 'dwpli'
            if data_key in mat_data:
                data = mat_data[data_key]
                if data.shape == (1, 1) and data.dtype.fields is not None:
                    if 'wpli_debiasedspctrm' in data.dtype.names:
                        wpli_data = data['wpli_debiasedspctrm'][0, 0]
                        for i in range(wpli_data.shape[2]):
                            slice_df = pd.DataFrame(wpli_data[:, :, i])
                            dataframes_wpli[f"{folder}_{mat_file}_wpli_debiasedspctrm_slice{i}"] = slice_df
                    else:
                        print(f"'wpli_debiasedspctrm' field not found in '{file_path}'")
                else:
                    print(f"Unexpected data structure in '{file_path}': {data.shape}, {data.dtype}")
            else:
                print(f"Data key '{data_key}' not found in '{file_path}'")
        else:
            print(f"File not found: {file_path}")

# dataframes now contains the extracted 'wpli_debiasedspctrm' data


Note that we have 40 different frequencies, we thus have 40 slices for each recording

In [ ]:
dataframes_wpli['p1_dwPLI_1_40Hz_wpli_debiasedspctrm_slice20']

# Some EDA to see what we have to work on

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from matplotlib import cm
data_slices = []
for i in range(40):
    data_slices += [dataframes_wpli[f'p1_dwPLI_1_40Hz_wpli_debiasedspctrm_slice{i}'].values.flatten(),]

# Number of bins for the histogram
n_bins = 20
colors = cm.viridis(np.linspace(0, 1, len(data_slices)))
plt.figure(figsize=(12, 8))
for i, data_slice in enumerate(data_slices):
    plt.hist(data_slice, bins=n_bins, alpha=0.5, color=colors[i], label=f'Slice {i+1}')

plt.title('Overlayed Histograms of Each Slice')
plt.xlabel('Value')
plt.ylabel('Frequency')
plt.legend()
plt.show()

plt.figure(figsize=(12, 8))
for i, data_slice in enumerate(data_slices):
    sns.kdeplot(data_slice, color=colors[i], label=f'Slice {i+1}')#, log_scale=(False,True))

plt.title('Density Plots of Each Slice')
plt.xlabel('Value')
plt.ylabel('Density')
plt.legend()
plt.show()

valid_slices = [s for s in data_slices if not np.all(np.isnan(s))]
if not valid_slices:
    raise ValueError("All slices are NaN")

global_min = min(np.nanmin(slice) for slice in valid_slices)
global_max = max(np.nanmax(slice) for slice in valid_slices)

# Create a 2D histogram array, handling NaN values within each slice
hist_data = []
for data_slice in data_slices:
    cleaned_slice = data_slice[~np.isnan(data_slice)]  # Remove NaN values from the slice
    hist, _ = np.histogram(cleaned_slice, bins=n_bins, range=(global_min, global_max))
    hist = np.where(hist == 0, np.nan, hist)
    hist_data.append(hist)

hist_data = np.array(hist_data)

plt.figure(figsize=(12, 8))
plt.imshow(hist_data, cmap=cm.viridis, aspect='auto', interpolation='nearest')
plt.colorbar(label='Frequency')
plt.title('Heatmap of Histogram Bins for Each Slice')
plt.xlabel('Bins')
plt.ylabel('Slice Number')
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.cm as cm

# Assuming 'folders_surv' and 'dataframes_wpli' are defined elsewhere in your code

# Setup for subplots
n_rows = 5
n_cols = 6
fig, axs = plt.subplots(n_rows, n_cols, figsize=(12 * n_cols, 8 * n_rows))
fig.subplots_adjust(hspace=0.4, wspace=0.4)  # Adjust spacing between plots

# Number of bins for the histogram
n_bins = 20

for px_idx, px in enumerate(folders_surv):
    data_slices = []
    for i in range(40):
        data_slices += [dataframes_wpli[f'{px}_dwPLI_1_40Hz_wpli_debiasedspctrm_slice{i}'].values.flatten()]

    valid_slices = [s for s in data_slices if not np.all(np.isnan(s))]
    if not valid_slices:
        raise ValueError("All slices are NaN")

    global_min = min(np.nanmin(slice) for slice in valid_slices)
    global_max = max(np.nanmax(slice) for slice in valid_slices)

    # Create a 2D histogram array, handling NaN values within each slice
    hist_data = []
    for data_slice in data_slices:
        cleaned_slice = data_slice[~np.isnan(data_slice)]  # Remove NaN values from the slice
        hist, _ = np.histogram(cleaned_slice, bins=n_bins, range=(global_min, global_max))
        hist = np.where(hist == 0, np.nan, hist)
        hist_data.append(hist)

    hist_data = np.array(hist_data)

    # Determine the position of the current subplot
    ax = axs[px_idx // n_cols, px_idx % n_cols]

    cax = ax.imshow(hist_data, cmap=cm.viridis, aspect='auto', interpolation='nearest')
    fig.colorbar(cax, ax=ax, label='Frequency')
    ax.set_title(f'{px}')
    ax.set_xlabel('Bins')
    ax.set_ylabel('Frequency+1 (in Hz)')

# Hide any unused subplots
for ax in axs.flatten()[len(folders_surv):]:
    ax.axis('off')

plt.show()


# Naive clustering from the wDPLI tensor using kmeans

## Cluster accross frequency and compare each patient for different number of clusters

In [ ]:
#do the same for all subjects, iterate over all subjects and store the cluster labels in a dictionary
from sklearn.metrics.cluster import adjusted_rand_score
from sklearn.metrics.cluster import normalized_mutual_info_score
import seaborn as sns
from sklearn.cluster import KMeans

n_electrodes = 63
n_clust = int(np.sqrt(n_electrodes))
fig, ax = plt.subplots(n_clust, 2, figsize=(12, 24), sharex=True, sharey=True)
plt.subplots_adjust(left=0.1, right=0.9, top=0.9, bottom=0.1, hspace=0.5, wspace=0.4)
for k in range(n_clust):
    cluster_labels_dict = {}
    for p in folders_surv:
        tensor_array = np.zeros((63,63, 40))
        for i in range(40):
            tensor_array[:,:,i] = dataframes_wpli[f'{p}_dwPLI_1_40Hz_wpli_debiasedspctrm_slice{i}'].values

        if np.isnan(tensor_array).any():
            tensor_array = np.nan_to_num(tensor_array)

        # Reshape the tensor into a 2D matrix (n x (n*t))
        reshaped_matrix = tensor_array.reshape(n_electrodes, -1)

        # Apply k-means clustering
        kmeans = KMeans(n_clusters=k+2, random_state=0).fit(reshaped_matrix)

        # Output the cluster labels for each node
        cluster_labels = kmeans.labels_
        cluster_labels_dict[p] = cluster_labels
    ARI_matrix = np.zeros((len(folders_surv),len(folders_surv)))
    NMI_matrix = np.zeros((len(folders_surv),len(folders_surv)))
    for i,p1 in enumerate(folders_surv):
        for j,p2 in enumerate(folders_surv):
            ARI_matrix[i,j] = adjusted_rand_score(cluster_labels_dict[p1], cluster_labels_dict[p2])
            NMI_matrix[i,j] = normalized_mutual_info_score(cluster_labels_dict[p1], cluster_labels_dict[p2])
    np.fill_diagonal(ARI_matrix, np.nan)
    np.fill_diagonal(NMI_matrix, np.nan)
    ari_min, ari_max = np.nanmin(ARI_matrix), np.nanmax(ARI_matrix)
    nmi_min, nmi_max = np.nanmin(NMI_matrix), np.nanmax(NMI_matrix)

    # Plotting with adjusted color bar range and colormap
    sns.heatmap(ARI_matrix, ax=ax[k, 0], vmin=ari_min, vmax=ari_max, cmap='binary', square=True)
    sns.heatmap(NMI_matrix, ax=ax[k, 1], vmin=nmi_min, vmax=nmi_max, cmap='binary', square=True)
ax[0,0].set_title('ARI')
ax[0,1].set_title('NMI')
#set title of the whole figure
fig.suptitle('Comparing clust accross freq between patients', fontsize=11)
plt.tight_layout()
plt.show()

## Cluster accross patients for each frequency using Kmeans

In [ ]:
#do the same for all subjects, iterate over all subjects and store the cluster labels in a dictionary
from sklearn.metrics.cluster import adjusted_rand_score
from sklearn.metrics.cluster import normalized_mutual_info_score
import seaborn as sns
from sklearn.cluster import KMeans

n_electrodes = 63
n_freq = 40
n_clust = int(np.sqrt(n_electrodes))
fig, ax = plt.subplots(n_clust, 2, figsize=(12, 24), sharex=True, sharey=True)
plt.subplots_adjust(left=0.1, right=0.9, top=0.9, bottom=0.1, hspace=0.5, wspace=0.4)
for k in range(n_clust):
    cluster_labels_dict = {}
    for freq in range(n_freq):
        tensor_array = np.zeros((63,63, len(folders_surv)))
        for j,p in enumerate(folders_surv):
            tensor_array[:,:,j] = dataframes_wpli[f'{p}_dwPLI_1_40Hz_wpli_debiasedspctrm_slice{freq}'].values

        if np.isnan(tensor_array).any():
            tensor_array = np.nan_to_num(tensor_array)

        # Reshape the tensor into a 2D matrix (n x (n*t))
        reshaped_matrix = tensor_array.reshape(n_electrodes, -1)

        # Apply k-means clustering
        kmeans = KMeans(n_clusters=k+2, random_state=0).fit(reshaped_matrix)

        # Output the cluster labels for each node
        cluster_labels = kmeans.labels_
        cluster_labels_dict[freq] = cluster_labels
    ARI_matrix = np.zeros((n_freq,n_freq))
    NMI_matrix = np.zeros((n_freq,n_freq))
    for i in range(n_freq):
        for j in range(n_freq):
            ARI_matrix[i,j] = adjusted_rand_score(cluster_labels_dict[i], cluster_labels_dict[j])
            NMI_matrix[i,j] = normalized_mutual_info_score(cluster_labels_dict[i], cluster_labels_dict[j])
    np.fill_diagonal(ARI_matrix, np.nan)
    np.fill_diagonal(NMI_matrix, np.nan)
    ari_min, ari_max = np.nanmin(ARI_matrix), np.nanmax(ARI_matrix)
    nmi_min, nmi_max = np.nanmin(NMI_matrix), np.nanmax(NMI_matrix)

    # Plotting with adjusted color bar range and colormap
    sns.heatmap(ARI_matrix, ax=ax[k, 0], vmin=ari_min, vmax=ari_max, cmap='binary', square=True)
    sns.heatmap(NMI_matrix, ax=ax[k, 1], vmin=nmi_min, vmax=nmi_max, cmap='binary', square=True)
ax[0,0].set_title('ARI Kmeans')
ax[0,1].set_title('NMI Kmeans')

plt.tight_layout()
plt.show()

# Naive clustering using spectral techniques

In [ ]:
from sklearn.cluster import SpectralClustering
n_electrodes = 63
n_freq = 40
n_clust = int(np.sqrt(n_electrodes))
fig, ax = plt.subplots(n_clust, 2, figsize=(12, 24), sharex=True, sharey=True)
plt.subplots_adjust(left=0.1, right=0.9, top=0.9, bottom=0.1, hspace=0.5, wspace=0.4)
for k in range(n_clust):
    cluster_labels_dict = {}
    for freq in range(n_freq):
        tensor_array = np.zeros((63, 63, len(folders_surv)))
        for j, p in enumerate(folders_surv):
            tensor_array[:, :, j] = dataframes_wpli[f'{p}_dwPLI_1_40Hz_wpli_debiasedspctrm_slice{freq}'].values

        if np.isnan(tensor_array).any():
            tensor_array = np.nan_to_num(tensor_array)

        # Reshape the tensor into a 2D matrix (n x (n*t))
        reshaped_matrix = tensor_array.reshape(n_electrodes, -1)

        # Perform spectral clustering
        spectral_clustering = SpectralClustering(n_clusters=k+2, affinity='nearest_neighbors', random_state=0)
        cluster_labels = spectral_clustering.fit_predict(reshaped_matrix)

        # Store the cluster labels for each frequency
        cluster_labels_dict[freq] = cluster_labels
    ARI_matrix = np.zeros((n_freq,n_freq))
    NMI_matrix = np.zeros((n_freq,n_freq))
    for i in range(n_freq):
        for j in range(n_freq):
            ARI_matrix[i,j] = adjusted_rand_score(cluster_labels_dict[i], cluster_labels_dict[j])
            NMI_matrix[i,j] = normalized_mutual_info_score(cluster_labels_dict[i], cluster_labels_dict[j])
    np.fill_diagonal(ARI_matrix, np.nan)
    np.fill_diagonal(NMI_matrix, np.nan)
    ari_min, ari_max = np.nanmin(ARI_matrix), np.nanmax(ARI_matrix)
    nmi_min, nmi_max = np.nanmin(NMI_matrix), np.nanmax(NMI_matrix)

    # Plotting with adjusted color bar range and colormap
    sns.heatmap(ARI_matrix, ax=ax[k, 0], vmin=ari_min, vmax=ari_max, cmap='binary', square=True)
    sns.heatmap(NMI_matrix, ax=ax[k, 1], vmin=nmi_min, vmax=nmi_max, cmap='binary', square=True)
ax[0,0].set_title('ARI Spectral')
ax[0,1].set_title('NMI Spectral')

plt.tight_layout()
plt.show()


## Clustering using all other methods

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics.cluster import adjusted_rand_score, normalized_mutual_info_score
import seaborn as sns
from sklearn.cluster import KMeans, SpectralClustering, AgglomerativeClustering, DBSCAN, OPTICS, MeanShift

# Define the clustering algorithms
clustering_algorithms = {
    'KMeans': KMeans,
    'SpectralClustering': SpectralClustering,
    'AgglomerativeClustering': AgglomerativeClustering,
    'DBSCAN': DBSCAN,
    'OPTICS': OPTICS,
    'MeanShift': MeanShift
}

def cluster_analysis(df, method, method_param, freq_list):
    fig, ax = plt.subplots(10, 2, figsize=(12, 24), sharex=True, sharey=True)
    plt.subplots_adjust(left=0.1, right=0.9, top=0.9, bottom=0.1, hspace=0.5, wspace=0.4)
    n_electrodes = df.shape[0]
    n_freq = len(freq_list)

    # Perform clustering for each algorithm and frequency
    for k in range(int(np.sqrt(n_electrodes))):
        cluster_labels_dict = {}
        for freq in freq_list:
            tensor_array = np.zeros((63, 63, len(folders_surv)))
            for j, p in enumerate(folders_surv):
                tensor_array[:, :, j] = df[f'{p}_dwPLI_1_40Hz_wpli_debiasedspctrm_slice{freq}'].values

            if np.isnan(tensor_array).any():
                tensor_array = np.nan_to_num(tensor_array)

            # Reshape the tensor into a 2D matrix (n x (n*t))
            reshaped_matrix = tensor_array.reshape(n_electrodes, -1)
            
            clustering = clustering_algorithms[method](n_clusters=k+2, **method_param)

            cluster_labels = clustering.fit_predict(reshaped_matrix)

            # Store the cluster labels for each algorithm and frequency
            if method not in cluster_labels_dict:
                cluster_labels_dict[method] = {}
            cluster_labels_dict[freq] = cluster_labels

            # Compute adjusted Rand index and normalized mutual information for each pair of algorithms
            ARI_matrix = np.zeros((n_freq,n_freq))
            NMI_matrix = np.zeros((n_freq,n_freq))
            for i in range(n_freq):
                for j in range(n_freq):
                    ARI_matrix[i,j] = adjusted_rand_score(cluster_labels_dict[i], cluster_labels_dict[j])
                    NMI_matrix[i,j] = normalized_mutual_info_score(cluster_labels_dict[i], cluster_labels_dict[j])
            np.fill_diagonal(ARI_matrix, np.nan)
            np.fill_diagonal(NMI_matrix, np.nan)
            ari_min, ari_max = np.nanmin(ARI_matrix), np.nanmax(ARI_matrix)
            nmi_min, nmi_max = np.nanmin(NMI_matrix), np.nanmax(NMI_matrix)

        # Plotting with adjusted color bar range and colormap
        sns.heatmap(ARI_matrix, ax=ax[k, 0], vmin=ari_min, vmax=ari_max, cmap='binary', square=True)
        sns.heatmap(NMI_matrix, ax=ax[k, 1], vmin=nmi_min, vmax=nmi_max, cmap='binary', square=True)

    # Set the titles for the subplots
    ax[0, 0].set_title('Adjusted Rand Index')
    ax[0, 1].set_title('Normalized Mutual Information')

    plt.tight_layout()
    plt.show()
    return fig


# Getting adjacency matrices from similarities

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, r"c:\users\arthu\documents\swittz-epfl\phd\code\lib\site-packages")
sys.path.append('../')
from pygraphon.estimators import SmoothNetHist

#from experiments.utils_generators import AssociativeFullyRandom
from pygraphon.norm import MseProbaEdge, MaeProbaEdge, AUCEdge, AUPRCEdge

In [ ]:
import numpy as np
# Function to create adjacency matrix based on threshold
def create_adjacency_matrix(similarity_matrix, threshold):
    if similarity_matrix.ndim != 2:
        raise ValueError("Input matrix must be 2-dimensional")
    adjacency_matrix = np.where(similarity_matrix > threshold, 1, 0)
    np.fill_diagonal(adjacency_matrix, 0)
    return adjacency_matrix

The following data_slices is by having each frequency in order, patient by patient. Thus the first patient corresponds to slice 0 to 39 where 0 is 1Hz and 39 is 40Hz.

In [ ]:
data_slices = []
for p in folders_surv:
    for i in range(40):
        data_slices += [dataframes_wpli[p+f'_dwPLI_1_40Hz_wpli_debiasedspctrm_slice{i}'].values,]

In [ ]:
print(len(folders_surv))

## Brute force threshold

In [ ]:
# Creating a dictionary to hold adjacency matrices for each percentile
adjacency_matrices_brute = {percentile: [] for percentile in [0.05*(x+1) for x in range(20)]}

# Looping through each percentile
for percentile in adjacency_matrices_brute:
    for slice in data_slices:
        if slice.ndim != 2:
            raise ValueError(f"Slice is not 2-dimensional: slice shape {slice.shape}")
        threshold = np.nanpercentile(slice, percentile)
        adjacency_matrix = create_adjacency_matrix(slice, threshold)
        adjacency_matrices_brute[percentile].append(adjacency_matrix)


In [ ]:
# Apply SmoothNetHist estimator to each adjacency matrix in the dictionary
estimators_brute = {percentile: [] for percentile in [0.05*(x+1) for x in range(20)]}
for percentile in adjacency_matrices_brute:
    for adjacency_matrix in adjacency_matrices_brute[percentile]:
        estimator = SmoothNetHist()
        estimator.fit(adjacency_matrix)
        estimators_brute[percentile].append(estimator)

## Threshold based on percentiles

In [ ]:
# Creating a dictionary to hold adjacency matrices for each percentile
adjacency_matrices_pct = {percentile: [] for percentile in range(10, 91, 10)}

# Looping through each percentile
for percentile in adjacency_matrices_pct:
    for slice in data_slices:
        if slice.ndim != 2:
            raise ValueError(f"Slice is not 2-dimensional: slice shape {slice.shape}")
        threshold = np.nanpercentile(slice, percentile)
        adjacency_matrix = create_adjacency_matrix(slice, threshold)
        adjacency_matrices_pct[percentile].append(adjacency_matrix)


In [ ]:
from pygraphon.plots.plot_matrices import spy
spy(adjacency_matrices_pct[90][120], marker_size=0.5)

In [ ]:
len(adjacency_matrices_pct[90])/40

## Compute modularity

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

# Define the community detection method
def community_detection(adjacency_matrix):
    # Create a graph from the adjacency matrix
    graph = nx.from_numpy_array(adjacency_matrix)

    # Perform community detection
    communities = nx.algorithms.community.greedy_modularity_communities(graph)

    # Return the modularity value
    return nx.algorithms.community.quality.modularity(graph, communities)

# Define the frequencies and thresholds
frequencies = range(1, 41)
thresholds = range(10, 91, 10)

# Initialize the modularity matrix
modularity_matrix = np.zeros((len(frequencies), len(thresholds)))

# Compute the modularity for each frequency and threshold
for i, frequency in enumerate(frequencies):
    for j, threshold in enumerate(thresholds):
        mod_per_slice = []
        for p in range(len(folders_surv)):
            # Get the adjacency matrix for the current frequency and threshold
            adjacency_matrix = adjacency_matrices_pct[threshold][p*i]

            # Compute the modularity
            mod_per_slice.append(community_detection(adjacency_matrix))

        # Store the modularity value in the matrix
        modularity_matrix[i, j] = np.mean(mod_per_slice)

# Plot the heatmap
plt.imshow(modularity_matrix, cmap='hot', aspect='auto')
plt.colorbar(label='Modularity')
plt.xlabel('Thresholding (Percentile)')
plt.ylabel('Frequency')
plt.xticks(range(len(thresholds)), thresholds)
plt.yticks(range(len(frequencies)), frequencies)
plt.title('Modularity Heatmap')
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

# Define the community detection method
def community_detection_clust(adjacency_matrix):
    # Create a graph from the adjacency matrix
    G = nx.from_numpy_array(adjacency_matrix)

    # Calculate the clustering coefficient
    clustering_coefficient = nx.average_clustering(G)

    # Return the modularity value
    return clustering_coefficient

# Define the frequencies and thresholds
frequencies = range(1, 41)
thresholds = range(10, 91, 10)

# Initialize the modularity matrix
modularity_matrix = np.zeros((len(frequencies), len(thresholds)))

# Compute the modularity for each frequency and threshold
for i, frequency in enumerate(frequencies):
    for j, threshold in enumerate(thresholds):
        mod_per_slice = []
        for p in range(len(folders_surv)):
            # Get the adjacency matrix for the current frequency and threshold
            adjacency_matrix = adjacency_matrices_pct[threshold][p*i]

            # Compute the modularity
            mod_per_slice.append(community_detection_clust(adjacency_matrix))

        # Store the modularity value in the matrix
        modularity_matrix[i, j] = np.mean(mod_per_slice)

# Plot the heatmap
plt.imshow(modularity_matrix, cmap='hot', aspect='auto')
plt.colorbar(label='Modularity')
plt.xlabel('Thresholding (Percentile)')
plt.ylabel('Frequency')
plt.xticks(range(len(thresholds)), thresholds)
plt.yticks(range(len(frequencies)), frequencies)
plt.title('Modularity Heatmap')
plt.show()



## SmoothNetHist to percentile adjacencies

In [ ]:
# Apply SmoothNetHist estimator to each adjacency matrix in the dictionary
estimators_pct = {percentile: [] for percentile in range(90, 91, 10)}
for percentile in estimators_pct:
    for adjacency_matrix in adjacency_matrices_pct[percentile]:
        estimator = SmoothNetHist()
        estimator.fit(adjacency_matrix)
        estimators_pct[percentile].append(estimator)

In [ ]:
estimators_pct[90][1].get_edge_connectivity()

In [ ]:
plt.imshow(estimators_pct[90][10].get_edge_connectivity(), cmap = plt.cm.binary)